# Analysis Notebook — SS-IDS Acceptance in Austria
## "No One Knows Which Individuals Will Represent the State in the Future"
### A Multi-Phase Expert-Informed Study on Organizational Acceptance of State-Sponsored Intrusion Detection Systems

This notebook contains the complete statistical analysis underpinning the survey findings
reported in the paper. Running all cells top-to-bottom reproduces every figure, table, and
statistic cited in the manuscript.

---

### Survey overview (N = 89 usable responses)

| # | Question block | Survey IDs | Paper section |
|---|---|---|---|
| RQ0 | Demographics | Q02, Q05, Q06, Q07, Q08, Q09, Q10 | §4 |
| RQ1 | Motivations & Perceptions | Q31, Q32, Q33, Q22, Q25, Q07/Q12 | §5.1 |
| RQ2 | Operational Preferences | Q23, Q26, Q27, Q17, Q37, Q29, Q30 | §5.2 |
| RQ3 | Trust & Security Concerns | Q39, Q34Copy, Q34, Q36, Q40, Q28 | §5.3 |

---

### Statistical approach

All pairwise comparisons use χ² tests (Pearson 1900). A single global
Benjamini–Hochberg FDR correction is applied across all 47 tests simultaneously.
Effect sizes are reported as Cramér's V. Groups with n < 10 are excluded from
inferential testing and reported descriptively only. Cochran's rule is verified
before each test; violations are noted inline. Post-hoc category identification
uses Haberman (1973) adjusted standardized residuals (|z| > 2, interpretive only).

5 confirmed findings (KF1–KF5) survive all corrections. All involve Q22 [S21] × Q34/Q36.

See `README.md` for setup instructions and data file locations.

---
## Section 0 . Imports & Setup

In [1]:
import re, itertools, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown

from scipy.stats import chi2_contingency, chi2
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

DATA_DIR        = "./survey"
SURVEY_CSV      = f"{DATA_DIR}/results-survey736756.csv"
TRANSLATION_CSV = f"{DATA_DIR}/sentences_translated.csv"

print(" Imports OK")


 Imports OK


---
## Section 1 . Data Loading & Cleaning

**Filters applied:**
- Attention check Q38 == 'Very secure'
- lastpage ≥ 2 (at least started the survey)
- Education label for HTL standardised to "Upper Secondary Technical School (HTL)"

In [2]:
data        = pd.read_csv(SURVEY_CSV,      sep=";")
translation = pd.read_csv(TRANSLATION_CSV, sep=";")

# ── Standardise HTL education label ─────────────────────────────────────────────
HTL_WRONG = "Higher technical school (e.g. Associate Degrees at a Community College)"
HTL_FIXED = "Upper Secondary Technical School (HTL)"
translation['text_en'] = translation['text_en'].replace(HTL_WRONG, HTL_FIXED)

def translate_text(source):
    if not isinstance(source, str): return source
    for _, t in translation[['text_de','text_en']].iterrows():
        if isinstance(t['text_de'], str) and source.strip() == t['text_de'].strip():
            return t['text_en']
    return source

# ── Column renaming ───────────────────────────────────────────────────────────
q_answer, q_dict, rename_dict = {}, {}, {}
for q in data.columns:
    q_id   = q.split('.')[0]
    q_long = '.'.join(q.split('.')[1:]).lstrip(' ')
    match  = re.match(r'^\$Q(.*)\[(.*)\]$', q_long)
    if match:
        q_answer[q_id] = match.groups()[1]
        q_long         = match.groups()[0]
    q_dict[q_id]    = q_long
    rename_dict[q]  = q_id
data.rename(columns=rename_dict, inplace=True)

q_r = re.compile(r'^Q([0-9]{0,3})(Copy|EXT){,1}(\[(.*)?\])?$')
all_questions = list(filter(q_r.match, data.columns))
for qid in all_questions:
    data[qid] = data[qid].apply(translate_text)

# ── Filter to usable data ─────────────────────────────────────────────────────
usable_data = data[data['Q38'] == 'Very secure'].copy()
usable_data = usable_data[usable_data['lastpage'] >= 2].copy()
total_part  = len(usable_data)

r_q = re.compile(r'^(?!.*(EXT|other|comment|Q35|Q24|Q99)).*$')
all_questions_q = list(filter(r_q.match, all_questions))

df_drop = {'q':  ['No response', 'Other:', 'Other'],
           'sq': ['No response', 'Other:', 'Other']}

print(f"Total raw responses:   {len(data)}")
print(f"Passed attention check: {len(data[data['Q38'] == 'Very secure'])}")
print(f"Usable (lastpage≥2):   {total_part}  <- N used in paper")
print()
print(f"HTL label now = '{HTL_FIXED}'")

Total raw responses:   192
Passed attention check: 92
Usable (lastpage≥2):   89  <- N used in paper

HTL label now = 'Upper Secondary Technical School (HTL)'


---
## Section 2 . Helper Functions (Visualization + Statistics)

In [3]:
# ── Visualization helpers ──────────────────────────────────────────────────────
order_dict_2 = {"Very secure":1, "Secure":2, "Insecure":3, "Very insecure":4}
order_dict_1 = {"No response":0, "Barely":1, "Neutral":2, "Strongly":3, "Fully":4}

def pr_info_get_df(qid, db=False, df=usable_data, drop_nr=False):
    df_s = df[qid]
    if isinstance(qid, list):
        for q in qid:
            if drop_nr: df_s = df_s[df_s[q].str.contains("No response")==False]
    else:
        if drop_nr: df_s = df_s[df_s.str.contains("No response")==False]
    return df_s, df_s.isna().sum()

def df_with_unique_values(df):
    return df.value_counts().rename_axis('unique_values').reset_index(name='counts')

def df_rel_counts(df, noans):
    s = df['counts'].sum()
    df['norm']  = (df['counts'] / s * 100).round(2)
    df['norm2'] = (df['counts'] / (s + noans) * 100).round(2)

def get_cross_stats(qid, qid2, db=False, df=usable_data, drop_nr=False, **kw):
    df_s, noans = pr_info_get_df([qid, qid2], db=db, df=df, drop_nr=drop_nr)
    df_s = df_s.value_counts().reset_index()
    total = df_s[[qid2, 'count']].groupby(qid2).sum().to_dict()['count']
    df_s['rel_count']   = df_s.apply(lambda x: round(x['count']/total[x[qid2]]*100,2), axis=1)
    df_s['grp_count']   = df_s.apply(lambda x: total[x[qid2]], axis=1)
    df_s['total_count'] = sum(total.values())
    return df_s, noans

def migrate_multiselection(qid, db=False, df=usable_data, drop_nr=False):
    sel_col = re.findall(rf"{qid}\[[0-9]+\]", ' '.join(q_dict.keys()))
    df_s, noans = pr_info_get_df(sel_col, df=df, db=db, drop_nr=drop_nr)
    new_names = []
    for col in sel_col:
        nm = re.findall(r'[a-zA-Z0-9 ]*\[([\w+0-9 ,()\ -/:]*?)\]', q_dict.get(col,''))
        new_names += nm
    if len(new_names) != len(sel_col): new_names = sel_col
    new_names = [translate_text(n) for n in new_names]  # translate sub-item labels
    new_df = df_s.set_axis(new_names, axis="columns")
    new_df = new_df.melt().groupby(['variable','value']).size().reset_index(name='Sizes')
    return new_df, noans

def get_bar(qid, qid2=None, grp=True, sort=True, color=0, x=0, y=0,
            db=False, rename=False, df=usable_data, drop_nr=False, **kw):
    if qid2:
        df_s, noans = get_cross_stats(qid, qid2, db=db, df=df, drop_nr=drop_nr)
        xc = q_dict.get(qid, qid)  if rename else qid
        cc = q_dict.get(qid2,qid2) if rename else qid2
        yc = 'rel_count' if not y else y
        if rename: df_s = df_s.rename(columns={qid: xc, qid2: cc})
        fig = px.bar(df_s, x=xc, y=yc, color=cc,
                     hover_data=["rel_count","count","grp_count","total_count"])
        if grp: fig.update_layout(barmode='group')
        if sort: fig.update_layout(xaxis={'categoryorder':'total descending'})
        return fig, df_s
    else:
        sel_col = re.findall(rf"{qid}\[[0-9]+\]", ' '.join(q_dict.keys()))
        if sel_col:
            new_df, noans = migrate_multiselection(qid, db=db, df=df, drop_nr=drop_nr)
            fig = px.bar(new_df, x='variable', y='Sizes', color='value')
            if sort: fig.update_layout(xaxis={'categoryorder':'total descending'})
            return fig, new_df
        else:
            df_s, noans = pr_info_get_df(qid, db=db, df=df, drop_nr=drop_nr)
            df_uv = df_with_unique_values(df_s)
            df_rel_counts(df_uv, noans)
            fig = px.bar(df_uv, x='unique_values', y='counts')
            if sort: fig.update_layout(xaxis={'categoryorder':'total descending'})
            return fig, df_uv

colwidth = 120
print(" Visualization helpers defined.")


 Visualization helpers defined.


In [4]:
# ── Statistical functions ───────────────────────────────────────────────────────

def check_cochrans_rule(contingency_df):
    "Verify Cochran's rule: all expected frequencies ≥1 and ≤20% below 5."
    if contingency_df.values.sum()==0 or len(contingency_df.columns)<=1:
        return False, False, False
    _, _, _, expected = chi2_contingency(contingency_df)
    exp_flat = expected.flatten()
    all_ge1  = bool(np.all(exp_flat >= 1))
    pct_ok   = bool((np.sum(exp_flat < 5)/len(exp_flat)) <= 0.20)
    return True, all_ge1, pct_ok

def test_phi_df(df, question_column, subquestion_column, alpha=0.05,
                drop=df_drop, db=False, min_cell_n=10):
    "χ² test with Cramér's V effect size, n≥10 group guard, and Cochran's rule check."
    for d in drop['q']:  df = df[df[question_column] != d]
    for d in drop['sq']: df = df[df[subquestion_column] != d]
    pivot = df.pivot_table(index=question_column, columns=subquestion_column,
                           values='count', aggfunc='sum').fillna(0)
    row_sums = pivot.sum(axis=1)
    small = row_sums[row_sums < min_cell_n]
    if len(small) > 0:
        if db: print(f"   n<{min_cell_n} guard: {question_column}x{subquestion_column} excluded groups {list(small.index)}")
        return None
    valid, all_ge1, pct_ok = check_cochrans_rule(pivot)
    if not valid: return None
    stat, p, dof, expected = chi2_contingency(pivot)
    N = int(pivot.values.sum())
    k = min(pivot.shape[0]-1, pivot.shape[1]-1)
    v = float(np.sqrt(stat/(N*k))) if (N>0 and k>0) else np.nan
    flag = "" if (all_ge1 and pct_ok) else " ⚠ Cochran violations present (see §3.3.3)"
    if db or p<=alpha:
        print(f"χ²({dof},N={N})={stat:.3f}, p={p:.4f}, V={v:.3f}{flag} | {question_column}x{subquestion_column}")
    return dict(question=question_column, subquestion=subquestion_column,
                chi2=stat, p=p, dof=dof, N=N, cramers_v=round(v,4),
                cochran_all_ge1=all_ge1, cochran_pct5_ok=pct_ok, significant=(p<=alpha))

def test_phi(question_column, subquestion_column, alpha=0.05,
             drop=df_drop, db=False, df=usable_data, min_cell_n=10):
    df_c = get_bar(question_column, subquestion_column, db=False, df=df)[1]
    return test_phi_df(df_c, question_column, subquestion_column,
                       alpha=alpha, drop=drop, db=db, min_cell_n=min_cell_n)

def pairwise_chi2_tests(dataframe, question_column, subquestion_column,
                        drop=df_drop, alpha=0.05, method='fdr_bh',
                        max_questions=20, db=False, min_cell_n=10):
    "Pairwise χ² tests with global FDR-BH correction, n≥10 guard, and Cochran's rule check."
    for d in drop['q']:  dataframe = dataframe[dataframe[question_column] != d]
    for d in drop['sq']: dataframe = dataframe[dataframe[subquestion_column] != d]
    unique = dataframe[question_column].unique()
    if len(unique) >= max_questions: return []
    pivot = dataframe.pivot_table(index=question_column, columns=subquestion_column,
                                  values='count', aggfunc='sum').fillna(0)
    p_values, comparisons = [], []
    for a1, a2 in itertools.combinations(unique, 2):
        tdf = pivot.loc[pivot.index.isin([a1, a2])]
        tdf = tdf.loc[:, (tdf != 0).any(axis=0)]
        if tdf.shape[0] < 2 or tdf.values.sum() == 0: continue
        N = int(tdf.values.sum())
        if N < min_cell_n:
            if db: print(f"   n<{min_cell_n} guard: {a1} vs {a2} excluded (N={N})")
            continue
        valid, all_ge1, pct_ok = check_cochrans_rule(tdf)
        if not valid: continue
        stat, p, dof, exp = chi2_contingency(tdf)
        p_values.append(p)
        comparisons.append((a1, a2, stat, p, dof, exp, N, all_ge1, pct_ok,
                             question_column, subquestion_column))
    if len(p_values) <= 1: return []
    reject, corrected, _, _ = multipletests(p_values, alpha=alpha, method=method)
    return list(zip(comparisons, reject, corrected))

def pairwise_chi2_tests_wr(q, sq, drop=df_drop, alpha=0.05,
                            method='fdr_bh', db=False, df=usable_data, min_cell_n=10):
    my_df = get_bar(q, sq, db=False, df=df)
    results = pairwise_chi2_tests(my_df[1], q, sq, drop=drop, alpha=alpha,
                                   method=method, db=db, min_cell_n=min_cell_n)
    return results, my_df

def batch_chi2_global_fdr(question_list, subquestion, alpha=0.05,
                          method='fdr_bh', df=usable_data, min_cell_n=10):
    "Collect raw p-values across a question list, then apply a single global FDR-BH pass."
    raw = []
    for q in question_list:
        df_b = get_bar(q, subquestion, db=False, df=df)[1]
        r = test_phi_df(df_b, q, subquestion, alpha=1.0, db=False, min_cell_n=min_cell_n)
        if r: raw.append(r)
    if not raw: return pd.DataFrame()
    pvals = [r['p'] for r in raw]
    reject, pvals_fdr, _, _ = multipletests(pvals, alpha=alpha, method=method)
    rows = []
    for r, rej, pfdr in zip(raw, reject, pvals_fdr):
        rows.append({'question':r['question'], 'subquestion':r['subquestion'],
                     'chi2':round(r['chi2'],3), 'p_raw':round(r['p'],5),
                     'p_fdr':round(pfdr,5), 'reject_H0':bool(rej),
                     'N':r['N'], 'cramers_v':r['cramers_v'],
                     'cochran_ok':(r['cochran_all_ge1'] and r['cochran_pct5_ok'])})
    return pd.DataFrame(rows).sort_values('p_raw')

def pr_pairwise_results(a_pair, alpha=0.05, label=""):
    results = a_pair[0]
    if not results: return
    found = False
    for comp, reject, cp in results:
        a1, a2, stat, p, dof, _, N, all_ge1, pct_ok, q, sq = comp
        if cp <= alpha:
            flag = " ⚠ Cochran violations present (see §3.3.3)" if not (all_ge1 and pct_ok) else ""
            print(f"   {a1} vs {a2}: χ²({dof})={stat:.3f}, p={p:.5f}, p_fdr={cp:.5f}, N={N}{flag}")
            found = True
    if not found:
        print(f"  No significant pairwise comparisons after FDR correction{' ('+label+')' if label else ''}.")

def get_driving_categories(contingency_df, z_threshold=2.0):
    """
    Post-hoc category selection via Haberman (1973) adjusted standardized residuals.

    Identifies which row×column cells most drive a significant χ² association,
    using Haberman's (1973) adjusted standardized residuals.

    The threshold |z| > z_threshold (default 2.0) is INTERPRETIVE, not inferential:
    no multiple-comparison correction is applied.  Only categories above the
    threshold are discussed in the main text (§3.3.3).

    Parameters
    ----------
    contingency_df : pd.DataFrame
        Contingency table (rows = row variable, columns = column variable).
        Must match the table passed to chi2_contingency (observed counts).
    z_threshold : float
        Absolute residual threshold for 'driving' flag (default 2.0).

    Returns
    -------
    pd.DataFrame
        All cells with |z| > z_threshold, sorted by descending |z|.
        Columns: row_category, col_category, observed, expected,
                 std_residual_z, abs_z, driving.
    """
    if contingency_df.values.sum() == 0:
        return pd.DataFrame()
    obs = contingency_df.values.astype(float)
    _, _, _, expected = chi2_contingency(contingency_df)
    N = obs.sum()
    row_sums = obs.sum(axis=1)
    col_sums = obs.sum(axis=0)
    rows_list = []
    for i, row_label in enumerate(contingency_df.index):
        for j, col_label in enumerate(contingency_df.columns):
            o = obs[i, j]
            e = expected[i, j]
            denom = np.sqrt(e * (1 - row_sums[i] / N) * (1 - col_sums[j] / N))
            z = (o - e) / denom if denom > 0 else np.nan
            rows_list.append({
                'row_category': row_label,
                'col_category': col_label,
                'observed': int(o),
                'expected': round(e, 2),
                'std_residual_z': round(z, 3) if not np.isnan(z) else np.nan,
                'abs_z': round(abs(z), 3) if not np.isnan(z) else np.nan,
                'driving': (abs(z) > z_threshold) if not np.isnan(z) else False,
            })
    result = pd.DataFrame(rows_list)
    driving = result[result['driving']].sort_values('abs_z', ascending=False).reset_index(drop=True)
    return driving


# compat aliases

print(" Statistical functions loaded.")

 Statistical functions loaded.


---
## Section 3 . Sample Overview & Dropout Analysis

**Context:** 89 usable responses out of a larger pool.
The high dropout rate is a known limitation (acknowledged in §3.4, C25).
We check whether early dropouts differ from completers on key variables.


In [5]:
# Response completion by survey page
ud_lp = usable_data.value_counts("lastpage").reset_index()
ud_lp.columns = ['lastpage', 'count']
fig_dropout = px.bar(ud_lp.sort_values('lastpage'), x='lastpage', y='count',
                     title="Responses per survey page (completers vs drop-offs)",
                     labels={'lastpage':'Survey page','count':'Respondents'})
fig_dropout.show()

print(f"Total usable N: {total_part}")
print()
# Response rates per question
resp_rates = usable_data[all_questions_q].count().reset_index()
resp_rates.columns = ['question','n_answered']
resp_rates['pct'] = (resp_rates['n_answered'] / total_part * 100).round(1)
print("Response rates per question (lowest 15):")
print(resp_rates.sort_values('pct').head(15).to_string(index=False))


Total usable N: 89

Response rates per question (lowest 15):
question  n_answered  pct
  Q25[6]           3  3.4
 Q26[28]           4  4.5
 Q18[16]           5  5.6
 Q17[12]           5  5.6
     Q20          12 13.5
 Q18[10]          32 36.0
  Q17[4]          36 40.4
  Q18[1]          36 40.4
 Q26[15]          36 40.4
  Q18[7]          37 41.6
  Q17[2]          37 41.6
 Q18[12]          37 41.6
 Q18[15]          38 42.7
  Q17[8]          38 42.7
 Q18[14]          39 43.8


---
## Section 4 - Demographics (RQ0)

Grouping variables for all cross-tabulation analyses.


### Q08 [S18] - Organization type (NIS2 classification)
>  **Survey question (S18):** "Does your organization fall into any of the following categories?"
>  *"Fällt ihre Organisation in einen der folgenden Bereiche?"*
>
> **Options:** Critical infrastructure / essential entities . Important entities . Public administration . None of the above
>
> **Paper use:** Primary grouping variable for all cross-tabs. Distinguishes "essential" (NIS2 Art. 3 critical sectors) from "important" entities.


In [6]:
fig_q08, df_q08 = get_bar('Q08', db=False)
fig_q08.update_layout(title="Q08: Organization type (NIS2 category)", xaxis_title="Category", yaxis_title="Count")
fig_q08.show()
print(df_q08[['unique_values','counts','norm']].to_string(index=False))


                                                                 unique_values  counts  norm
Critical infrastructure, essential facilities, operators of essential services      26 29.21
                                                                            No      25 28.09
                            Public administration and government organisations      20 22.47
                                   Important organisation according to NIS 2.0      12 13.48
                                                                   No response       6  6.74


### Q09 [S19] - Organization size
>  **Survey question (S19):** "How many people work in your organisation?"
>  *"Wie viele Personen arbeiten in Ihrer Organisation?"*
>
> **Note:** Smaller size categories (1, 2–9, 10–49) have n<10 in many cross-tabs —
> inferential testing is excluded; only descriptive statistics are reported.


In [7]:
fig_q09, df_q09 = get_bar('Q09', db=False)
fig_q09.update_layout(title="Q09: Organization size")
fig_q09.show()
print(df_q09[['unique_values','counts','norm']].to_string(index=False))


unique_values  counts  norm
        > 249      59 66.29
     50 – 249      16 17.98
      10 – 49       6  6.74
        2 - 9       4  4.49
  No response       3  3.37
            1       1  1.12


### Q06 [S16] - Role / position
>  **Survey question (S16):** "What position do you currently hold in your current organisation?"
>  *"Welche Position haben Sie aktuell in Ihrer derzeitigen Organisation inne?"*


In [8]:
fig_q06, df_q06 = get_bar('Q06', db=False)
fig_q06.update_layout(title="Q06: Respondent role")
fig_q06.show()
print(df_q06[['unique_values','counts']].to_string(index=False))


                                                     unique_values  counts
Employee with decision-making authority in the area of IT security      37
                                                          Employee      35
                                                       No response       7
                                                            Other:       5
                                                  Management Board       5


### Q10 [S20] - Industry sector
>  **Survey question (S20):** "In which industry do you work?"
>  *"In welcher Branche arbeiten Sie?"*
>
> **Note (C29):** "Life" corrected to "Life Sciences / Biotechnology" in the revised appendix.
> **Note:** "HTL" education items relabelled to Upper Secondary Technical School.


In [9]:
fig_q10, df_q10 = get_bar('Q10', db=False)
fig_q10.update_layout(title="Q10: Industry sector")
fig_q10.show()
print(df_q10[['unique_values','counts']].to_string(index=False))


                                           unique_values  counts
                                 Telecommunications & IT      20
Economy & Politics (Public Administration, Defense, ...)      12
                        Finance, Insurance & Real Estate       9
                                                  Other:       9
                   Society (Education, Research, Law, …)       7
                                             No response       6
                                                 Schools       4
                                    Energy & Environment       4
                                   Transport & Logistics       3
                               Chemicals & Raw Materials       3
                                            Construction       3
                                     Metal & Electronics       3
                                Pharmaceuticals & Health       2
                                                Internet       2
                         

### Q07 [S17] - Self-rated knowledge of detection/response systems
>  **Survey question (S17):** "How would you rate your knowledge in the area of detection and response systems?"
>  *"Wie schätzen Sie ihre Kenntnisse im Bereich von Detektions- und Erkennungssystemen ein?"*
>
> **Paper use:** Used as a control - does knowledge predict willingness or mental model?


In [10]:
fig_q07, df_q07 = get_bar('Q07', db=False)
fig_q07.update_layout(title="Q07: Self-rated IDS knowledge")
fig_q07.show()
print(df_q07[['unique_values','counts','norm']].to_string(index=False))


         unique_values  counts  norm
         Knowledgeable      36 40.45
Slightly knowledgeable      19 21.35
 Average knowledgeable      18 20.22
                Expert      13 14.61
           No response       2  2.25
          No knowledge       1  1.12


### Q02 [S13] - Age group
>  **Survey question (S13):** "Age group"
>  *"Altersgruppe"*


In [11]:
fig_q02, df_q02 = get_bar('Q02', db=False)
fig_q02.update_layout(title="Q02: Age group")
fig_q02.show()


### Q11 [S33] - Current use of security monitoring systems
>  **Survey question (S33):** "Does your organization currently use security monitoring systems?"
>  Existing deployment of detection/response solutions.


In [12]:
fig_q11, df_q11 = get_bar('Q11', db=False)
fig_q11.update_layout(title="Q11: Current use of security monitoring systems")
fig_q11.show()
print(df_q11[['unique_values','counts','norm']].to_string(index=False))


unique_values  counts  norm
          Yes      48 76.19
   Don't know       6  9.52
           No       5  7.94
  No response       4  6.35


### Q12 [S34] - Understanding of what an IDS actually does
>  **Survey question (S34):** "How would you describe your understanding of what an intrusion detection system actually does?"
>  *"Wie würden Sie Ihr Verständnis beschreiben was ein Intrusion Detection System wirklich macht?"*
>
> **Paper use (§5.1):** Identifies the "flawed mental model" problem -
> many respondents believe IDS actively blocks attacks rather than detecting them.


In [13]:
fig_q12, df_q12 = get_bar('Q12', db=False)
fig_q12.update_layout(title="Q12: Understanding of IDS function")
fig_q12.show()
print(df_q12[['unique_values','counts','norm']].to_string(index=False))

# Cross: Q12 x Q07 (knowledge vs. mental model)
print("\n--- Q12 x Q07: Does better self-rated knowledge = better mental model? ---")
fig_q12_q07, _ = get_bar('Q12', 'Q07')
fig_q12_q07.update_layout(title="Q12 x Q07: IDS understanding vs. self-rated knowledge")
fig_q12_q07.show()


                                                                                       unique_values  counts  norm
We use them to identify potential threats and vulnerabilities at an early stage and counteract them.      27 43.55
                                      We use them to prevent attackers from penetrating our systems.      18 29.03
                                   We use them to detect attackers on our systems and prevent worse.      15 24.19
                                     We use them to minimize the risk of our backup systems failing.       2  3.23

--- Q12 x Q07: Does better self-rated knowledge = better mental model? ---


---
## Section 5 - RQ1 - Motivations & Perceptions (§5.1)

RQ1: How is IDS importance perceived? Willingness to participate in state-sponsored IDS? Expected benefits?


### Q31 [S1] - Importance of detection and response systems (general)
>  **Survey question (S1):** "How important do you consider detection and response systems?"
>  *"Wie wichtig schätzen Sie Erkennungs- und Reaktionssysteme ein?"*
>
> **Scale:** Not important . Neutral . Important . Very important
> **Paper use (§5.1):** Establishes baseline - do respondents already value D&R before we ask about state involvement?


In [14]:
fig_q31, df_q31 = get_bar('Q31', db=False)
fig_q31.update_layout(title="Q31: Importance of D&R systems (N=89)")
fig_q31.show()
print(df_q31[['unique_values','counts','norm']].to_string(index=False))

# Cross-tab: Q31 x Q08 (by org type)
print("\n--- Q31 x Q08: Does entity type predict perceived importance? ---")
r31_q08 = test_phi('Q31', 'Q08', alpha=0.05, min_cell_n=10)
if r31_q08:
    print(f"χ²({r31_q08['dof']},N={r31_q08['N']})={r31_q08['chi2']:.3f}, p={r31_q08['p']:.4f}, V={r31_q08['cramers_v']:.3f}")
    pr_pairwise_results(pairwise_chi2_tests_wr('Q31','Q08', alpha=0.05, min_cell_n=10), 0.05)


 unique_values  counts  norm
Very important      70 78.65
     Important      16 17.98
       Neutral       2  2.25
 Not important       1  1.12

--- Q31 x Q08: Does entity type predict perceived importance? ---


### Q32 [S2] - Does using such a system increase security?
>  **Survey question (S2):** "Do you think the use of such a system increases security?"
>  *"Denken Sie der Einsatz eines solchen Systems steigert die Sicherheit?"*
>
> **Scale:** Not important . Neutral . Important . Very important
> **Paper use (§5.1):** Measures general belief in IDS effectiveness - a prerequisite for adoption willingness.


In [15]:
fig_q32, df_q32 = get_bar('Q32', db=False)
fig_q32.update_layout(title="Q32: Does an IDS increase security? (N=89)")
fig_q32.show()
print(df_q32[['unique_values','counts','norm']].to_string(index=False))

# Combined Q31/Q32/Q33 stacked bar (paper Fig. X)
import plotly.graph_objects as go

pltq31 = get_bar('Q31',db=False)[1][['unique_values','counts']].rename(columns={'unique_values':'value','counts':'count'})
pltq31['variable']='Q31: Importance'
pltq32 = get_bar('Q32',db=False)[1][['unique_values','counts']].rename(columns={'unique_values':'value','counts':'count'})
pltq32['variable']='Q32: Increases security'
pltq33 = get_bar('Q33',db=False)[1][['unique_values','counts']].rename(columns={'unique_values':'value','counts':'count'})
pltq33['variable']='Q33: Gain from existing use'
plot_data = pd.concat([pltq31, pltq32, pltq33])
fig_rq1_combined = px.bar(plot_data, x='variable', y='count', color='value', barmode='group',
    title="RQ1 Combined: Importance . Security increase . Benefit from existing use")
fig_rq1_combined.show()


 unique_values  counts  norm
Very important      47 52.81
     Important      39 43.82
       Neutral       2  2.25
 Not important       1  1.12


### Q33 [S3] - Benefit from existing D&R system use
>  **Survey question (S3):** "By operating detection and response systems, my organization has gained benefit."
>  *"Durch den Betrieb von Erkennungs- und Reaktionssystemen hat meine Organisation einen Nutzen erwirtschaftet."*
>
> **Paper use (§5.1):** Only asked to organizations that already have IDS deployed (Q11 filter).
> Tests whether actual experience with IDS correlates with perceived benefit.


In [16]:
fig_q33, df_q33 = get_bar('Q33', db=False)
fig_q33.update_layout(title="Q33: Benefit gained from existing D&R systems")
fig_q33.show()
print(df_q33[['unique_values','counts','norm']].to_string(index=False))


             unique_values  counts  norm
                 Important      34 38.20
            Very important      22 24.72
               No response      14 15.73
                   Neutral       8  8.99
We do not use such systems       8  8.99
             Not important       3  3.37


### Q22 [S21] - Willingness to participate in state-sponsored IDS
>  **Survey question (S21, S3):** "Would your organization participate in a government-provided detection and response system?"
>  *"Würde Ihre Organisation an einem staatlich zur Verfügung gestellten Erkennungs- und Reaktionssystem teilnehmen?"*
>
> **Scale:** Yes . No . Don't know
>
>  **This is the KEY OUTCOME VARIABLE for the paper.**
> The 5 confirmed statistical findings all involve Q22 as one of the variables.
>
> **C22 Fix:** 45.21% is a *plurality*, not a majority - corrected in revised paper.


In [17]:
fig_q22, df_q22 = get_bar('Q22', db=False)
fig_q22.update_layout(title="Q22: Willingness to participate in state SS-IDS (N=89)")
fig_q22.show()
print(" Distribution:")
print(df_q22[['unique_values','counts','norm']].to_string(index=False))

# Open text responses
print("\n Open-text reasons (Q22[other]):")
colwidth = 200
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q22[other]'].notna()]['Q22[other]']
    for i, v in enumerate(ot.values):
        print(f"  [{i+1}] {v}")


 Distribution:
unique_values  counts  norm
   Don't know      33 45.21
          Yes      19 26.03
           No      14 19.18
       Other:       4  5.48
  No response       3  4.11

 Open-text reasons (Q22[other]):
  [1] abhängig von Kosten, Professionalität und Einfachheit 
  [2] Vermutlich ja
  [3] Aus meiner Sicht auf keinen Fall
  [4] Aus meiner Persönlich Sicht NEIN


### Q25 [S24] - Expected benefits from a state SS-IDS
>  **Survey question (S24):** "What does your organization expect to gain from such a system?"
>  *"Was erwartet sich Ihre Organisation von einem solchen System?"*
>
> **Options (multi-select):**
> [1] Better internal information sharing
> [2] Better situational awareness across sectors
> [3] Additional external expertise
> [4] Communication platform for rapid cross-org exchange
> [5] Access to affordable Threat Intelligence feeds


In [18]:
fig_q25, df_q25 = get_bar('Q25', db=False)
fig_q25.update_layout(title="Q25: Expected benefits from state SS-IDS (multi-select)")
fig_q25.show()

# Filter out 'Other' for cleaner chart
plot_data_q25 = df_q25[df_q25['variable'] != 'Other (please specify)']
fig_q25b = px.bar(plot_data_q25, x='variable', y='Sizes', color='value',
    title="Q25: Expected benefits (excl. Other)")
fig_q25b.update_layout(xaxis={'categoryorder':'total descending'})
fig_q25b.show()


---
## Section 6 - RQ2 - Operational Preferences (§5.2)

RQ2: What are the operational preferences for IDS adoption?

Note: All subgroup comparisons involving entity type are excluded by the n<10 guard. RQ2 is answered descriptively only.


### Q23 [S22] - Which services would be used?
>  **Survey question (S22):** "Which of the following activities would your organization use?"
>  *"Welche der folgenden Aktivitäten würde Ihre Organisation beziehen?"*
>
> **Options:**
> [1] Detection (Erkennung)
> [2] Response (Reaktion)


In [19]:
fig_q23, df_q23 = get_bar('Q23', db=False)
fig_q23.update_layout(title="Q23: Which services would be used (detection vs response)?")
fig_q23.show()

# Q23 x Q07 (knowledge vs service preference)
fig_q23_q07, _ = get_bar('Q23[1]', 'Q07', db=False)
fig_q23_q07.update_layout(title="Q23[1] Detection x Q07 Knowledge level")
fig_q23_q07.show()

fig_q23_q07b, _ = get_bar('Q23[2]', 'Q07', db=False)
fig_q23_q07b.update_layout(title="Q23[2] Response x Q07 Knowledge level")
fig_q23_q07b.show()


### Q26 [S26] - Participation requirements (multi-select)
>  **Survey question (S26):** "What would be the requirements for your organization to participate?"
>  *"Was wären die Rahmenbedingungen Ihrer Organisation, um teilzunehmen?"*
>
> **Key sub-items (most relevant for paper):**
> [2] Feedback channel & co-creation opportunity
> [6] Operator is a government agency
> [7] Operator is a private / commercial entity
> [8] State oversight of the operator
> [12] Control over collected data
> [17] Data stays within the organization
> [18] Anonymization of personal data
>
> **Note:** A single global FDR-BH correction is applied across all sub-items (see Section 8).


In [20]:
fig_q26, df_q26 = get_bar('Q26', db=False)
fig_q26.update_layout(title="Q26: Participation requirements (multi-select, ordered by importance)")
fig_q26.update_layout(xaxis={'categoryorder':'total descending'})
fig_q26.show()

# Global FDR-BH batch test: Q26[i] x Q08 (entity type)
q26_questions = [f'Q26[{i}]' for i in range(1, 28) if i != 24]
print("\n=== Q26[i] x Q08 — Global FDR-BH correction (min n≥10) ===")
res_q26_q08 = batch_chi2_global_fdr(q26_questions, 'Q08', alpha=0.05, min_cell_n=10)
if len(res_q26_q08):
    sig = res_q26_q08[res_q26_q08['reject_H0']]
    print(f"Significant after global FDR-BH: {len(sig)}/{len(res_q26_q08)}")
    display(res_q26_q08[['question','p_raw','p_fdr','reject_H0','N','cramers_v']].head(10))
else:
    print("No testable pairs (n<10 guards applied)")



=== Q26[i] x Q08 — Global FDR-BH correction (min n≥10) ===


χ²(6,N=43)=15.325, p=0.0179, V=0.422 ⚠ Cochran violations present (see §3.3.3) | Q26[6]xQ08


χ²(6,N=42)=2.689, p=0.8467, V=0.179 ⚠ Cochran violations present (see §3.3.3) | Q26[14]xQ08


Significant after global FDR-BH: 1/2


,question,p_raw,p_fdr,reject_H0,N,cramers_v
0,Q26[6],0.01788,0.03575,True,43,0.4221
1,Q26[14],0.84669,0.84669,False,42,0.1789


### Q26[2] x Q09 - Required exclusion criterion x organization size

> This sub-item is discussed in §5.2 of the paper. After global FDR correction, the association does not survive (n<10 guard applies).


In [21]:
print("=== Q26[2] x Q09: 'Required exclusion criterion' x org size ===")
r_q26_q09 = test_phi('Q26[2]', 'Q09', alpha=0.05, min_cell_n=10)
if r_q26_q09:
    print(f"Overall χ²({r_q26_q09['dof']},N={r_q26_q09['N']})={r_q26_q09['chi2']:.3f}, "
          f"p={r_q26_q09['p']:.4f}, V={r_q26_q09['cramers_v']:.3f}")
    pr_pairwise_results(pairwise_chi2_tests_wr('Q26[2]','Q09', alpha=0.05, min_cell_n=10), 0.05)
    fig_q26_q09, _ = get_bar('Q26[2]', 'Q09')
    fig_q26_q09.update_layout(title="Q26[2] (Required exclusion) x Q09 (org size)")
    fig_q26_q09.show()
else:
    print(" n<10 guard: subgroups too small for inferential testing. Descriptive only:")
    fig_q26_q09, df_q26_q09 = get_bar('Q26[2]', 'Q09')
    fig_q26_q09.update_layout(title="Q26[2] x Q09 (descriptive - no significance claimed)")
    fig_q26_q09.show()


=== Q26[2] x Q09: 'Required exclusion criterion' x org size ===
 n<10 guard: subgroups too small for inferential testing. Descriptive only:


### Q27 [S28] - Preferred operator type (state vs commercial)
>  **Survey question (S28):** "Would you prefer the system to be operated by a state or commercial entity?"
>  *"Würden Sie einen staatlichen oder kommerziellen Betrieb bevorzugen?"*


In [22]:
fig_q27, df_q27 = get_bar('Q27', db=False)
fig_q27.update_layout(title="Q27: Preferred operator type")
fig_q27.show()
print(df_q27[['unique_values','counts','norm']].to_string(index=False))

# Q27 x Q26[6] (prefers state operator x requires state operator)
fig_q27_q08, _ = get_bar('Q27', 'Q08')
fig_q27_q08.update_layout(title="Q27 x Q08: Operator preference by entity type")
fig_q27_q08.show()


unique_values  counts  norm
           No      41 78.85
          Yes      11 21.15


### Q17 [S40] - MSSP selection criteria (general importance ratings)
>  **Survey question (S40):** "How does your organization rate the importance of the following criteria when selecting an MSSP?"
>  *"Wie bewertet Ihre Organisation die Wichtigkeit der folgenden Kriterien bei der Auswahl eines MSSPs?"*
>
> **Scale:** Not relevant . Low . Medium . High . Essential
>
> **Paper use:** Baseline MSSP preferences - shows what organizations prioritize generally,
> which can inform SS-IDS design requirements.


In [23]:
fig_q17, df_q17 = get_bar('Q17', grp=False, db=False)
fig_q17.update_layout(title="Q17: MSSP selection criteria importance (multi-item)")
fig_q17.update_layout(xaxis={'categoryorder':'total descending'})
fig_q17.show()


### Q37 [S9] - Where should the IDS be operated?
>  **Survey question (S9):** "Where should the operation of the detection and response systems be located?"
>  *"Wo soll der Betrieb der Erkennungs- und Reaktionssysteme angesiedelt sein?"*
>
> **Options:** Within my organization . By the operator . In a state datacenter . Hybrid . No preference


In [24]:
fig_q37, df_q37 = get_bar('Q37', db=False)
fig_q37.update_layout(title="Q37: Where should the IDS be operated?")
fig_q37.show()
print(df_q37[['unique_values','counts','norm']].to_string(index=False))

# Q37 x Q08 (location preference by entity type)
fig_q37_q08, _ = get_bar('Q37', 'Q08')
fig_q37_q08.update_layout(title="Q37 x Q08: Operation location preference by entity type")
fig_q37_q08.show()

# Open text
print("\n Other responses (Q37[other]):")
with pd.option_context('display.max_colwidth', 200):
    ot = usable_data[usable_data['Q37[other]'].notna()]['Q37[other]']
    for v in ot.values: print(f"  - {v}")


                                                                 unique_values  counts  norm
                                 Association or group (e.g. CERT, sector CERT)      29 32.58
                                                                  Governmental      23 25.84
Private organisation on behalf of and under the control of a government agency      14 15.73
                                                                        Other:      10 11.24
                                                        At European / EU level       9 10.11
                                                                   No response       4  4.49



 Other responses (Q37[other]):
  - egal
  - Zusammenarbeit aus allen
  - intern
  - EU + Staatlich kontrollierte private Organisationen
  - Spezielle Dienstleistungsunternehmen, In großen Konzernen konzerninter, für behörden durch eine staatliche Einrichgung.
  - kommerzielle anbieter
  - Die Daten beim Unternehmen und als Meldestelle das Sektorale CERT
  - Nur IOCs sollten an ein Sektor-Cert weitergeleitet werden.
  - Inhouse
  - Privat, sofern streng reglementiert und Missbrauch zu verhindern, Alternativ Verein/Verbund


### Q29 [S31] - Where should data processing / storage take place?
>  **Survey question (S31):** "Where should data processing and data storage take place?"
>  *"Wo soll die Datenverarbeitung und -speicherung stattfinden?"*


In [25]:
fig_q29, df_q29 = get_bar('Q29', grp=False, db=False)
fig_q29.update_layout(title="Q29: Where should data be processed/stored?")
fig_q29.show()

# Q29 x Q12 (data location preference vs. understanding of IDS)
fig_q29_q12, _ = get_bar('Q29', 'Q12', grp=False)
fig_q29_q12.update_layout(title="Q29 x Q12: Data location pref x IDS understanding")
fig_q29_q12.show()


### Q30 [S32] - Who should operate the system?
>  **Survey question (S32):** "Who should operate such a system?"
>  *"Wer soll ein solches System betreiben?"*


In [26]:
fig_q30, df_q30 = get_bar('Q30', db=False)
fig_q30.update_layout(title="Q30: Who should operate the system?")
fig_q30.show()
print(df_q30[['unique_values','counts','norm']].to_string(index=False))


                                              unique_values  counts  norm
Permanent access, e.g. through direct access to the systems      22 30.99
        Direct, time-limited access after manual activation      19 26.76
Made available on request via exports or comparable methods      10 14.08
                                                No response       9 12.68
                                                  No access       8 11.27
                                            Is not relevant       2  2.82
                                                     Other:       1  1.41


### Q15 [S38] - Current use of external security service providers
>  **Survey question (S38):** "Does your organization currently use external security service providers?"
>
> **Paper use:** Baseline - if an org already outsources security,
> they may be more comfortable with state-operated SS-IDS.


In [27]:
fig_q15, df_q15 = get_bar('Q15[1]', grp=True, db=False)
fig_q15.update_layout(title="Q15[1]: Uses external SOC/MSSP provider?")
fig_q15.show()

fig_q15b, _ = get_bar('Q11', 'Q15[1]', grp=True)
fig_q15b.update_layout(title="Q11 x Q15[1]: Has IDS x Uses external provider")
fig_q15b.show()


---
## Section 7 - RQ3 - Trust & Security Concerns (§5.3)

RQ3: Are IDS seen as privacy threats? How is government involvement perceived?

All 5 confirmed statistical findings are in this section (Q22 [S21] x Q34/Q36 pairs).


### Q39 [S11] - Feeling of personal surveillance
>  **Survey question (S11):** "Do you personally feel more monitored since your organization uses detection and response systems?"
>  *"Fühlen Sie sich persönlich mehr überwacht, seitdem Ihre Organisation Erkennungs- und Reaktionssysteme einsetzt?"*
>
> **Paper use:** Establishes baseline surveillance perception - a precondition for trust concerns.


In [28]:
fig_q39, df_q39 = get_bar('Q39', db=False)
fig_q39.update_layout(title="Q39: Do you feel monitored since using D&R systems?")
fig_q39.show()
print(df_q39[['unique_values','counts','norm']].to_string(index=False))


             unique_values  counts  norm
                        No      50 56.18
We do not use such systems      18 20.22
                       Yes      11 12.36
               No response      10 11.24


### Q34Copy [S4] - Feelings about STATE-operated service
>  **Survey question (S4):** "How would you feel if a government agency operates such a service for your organization?"
>  *"Wie würden Sie sich fühlen, wenn eine staatliche Stelle einen solchen Dienst für Ihre Organisation betreibt?"*
>
> **Scale:** Very insecure . Insecure . Secure . Very secure
>
>  **Q34Copy x Q22 = CONFIRMED FINDING #1** (cor_p=0.002, V=0.50 - large effect)
> Organizations uncomfortable with state operation are significantly less willing to participate.


In [29]:
fig_q34c, df_q34c = get_bar('Q34Copy', db=False)
fig_q34c.update_layout(title="Q34Copy: Feelings about state-operated SS-IDS")
fig_q34c.show()
print(df_q34c[['unique_values','counts','norm']].to_string(index=False))

#  CONFIRMED FINDING #1: Q34Copy x Q22
print("\n CONFIRMED FINDING #1: Q34Copy x Q22")
print("Question: Do feelings about state operation predict willingness to participate?")
r1 = test_phi('Q22', 'Q34Copy', alpha=0.05, min_cell_n=10)
if r1:
    print(f"χ²({r1['dof']}, N={r1['N']}) = {r1['chi2']:.3f}")
    print(f"p = {r1['p']:.5f}")
    print(f"Cramér's V = {r1['cramers_v']:.3f}  (LARGE effect, V≥0.4)")
    pr_pairwise_results(pairwise_chi2_tests_wr('Q22','Q34Copy', alpha=0.05, min_cell_n=10), 0.05)

fig_q22_q34c, _ = get_bar('Q22', 'Q34Copy')
fig_q22_q34c.update_layout(title="Q22 x Q34Copy: Willingness x Feelings about state operation ")
fig_q22_q34c.show()


unique_values  counts  norm
       Secure      34 38.20
     Insecure      24 26.97
  Very secure      13 14.61
Very insecure      11 12.36
  No response       7  7.87

 CONFIRMED FINDING #1: Q34Copy x Q22
Question: Do feelings about state operation predict willingness to participate?
χ²(6,N=62)=31.546, p=0.0000, V=0.504 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34Copy
χ²(6, N=62) = 31.546
p = 0.00002
Cramér's V = 0.504  (LARGE effect, V≥0.4)
   Don't know vs No: χ²(3)=22.758, p=0.00005, p_fdr=0.00014, N=44 ⚠ Cochran violations present (see §3.3.3)
   Yes vs No: χ²(3)=20.148, p=0.00016, p_fdr=0.00024, N=32 ⚠ Cochran violations present (see §3.3.3)


### Q34 [S5] - Feelings about COMMERCIAL operator (state-contracted)
>  **Survey question (S5):** "How would you feel if a commercial entity in a government contract operates such a service for your organization?"
>  *"Wie würden Sie sich fühlen, wenn eine kommerzielle Stelle im staatlichen Auftrag einen solchen Dienst betreibt?"*
>
>  **Q34 x Q22 = CONFIRMED FINDING #2** (cor_p=0.005, V=0.43 - medium-large effect)
> Comparison Q34 vs Q34Copy shows whether state vs. commercial operation differs.


In [30]:
fig_q34, df_q34 = get_bar('Q34', grp=False, db=False)
fig_q34.update_layout(title="Q34: Feelings about commercially-operated (state-contracted) SS-IDS")
fig_q34.show()
print(df_q34[['unique_values','counts','norm']].to_string(index=False))

# Q34 vs Q34Copy direct comparison
fig_q34_q34c, _ = get_bar('Q34Copy', 'Q34', grp=False)
fig_q34_q34c.update_layout(title="Q34Copy vs Q34: State-operated vs commercially-operated feelings")
fig_q34_q34c.show()

#  CONFIRMED FINDING #2: Q34 x Q22
print("\n CONFIRMED FINDING #2: Q34 x Q22")
r2 = test_phi('Q22', 'Q34', alpha=0.05, min_cell_n=10)
if r2:
    print(f"χ²({r2['dof']}, N={r2['N']}) = {r2['chi2']:.3f}")
    print(f"p = {r2['p']:.5f}, Cramér's V = {r2['cramers_v']:.3f}")
    pr_pairwise_results(pairwise_chi2_tests_wr('Q22','Q34', alpha=0.05, min_cell_n=10), 0.05)

# Open text: Q35 (why do you feel that way?)
print("\n Q35 - Why do you feel that way (open text)?")
with pd.option_context('display.max_colwidth', 200):
    ot = usable_data[usable_data['Q35'].notna()]['Q35']
    for i, v in enumerate(ot.values):
        print(f"  [{i+1}] {v}")


unique_values  counts  norm
       Secure      37 41.57
     Insecure      22 24.72
Very insecure      13 14.61
  No response       9 10.11
  Very secure       8  8.99



 CONFIRMED FINDING #2: Q34 x Q22
χ²(6,N=60)=21.690, p=0.0014, V=0.425 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34
χ²(6, N=60) = 21.690
p = 0.00138, Cramér's V = 0.425


   Don't know vs No: χ²(3)=13.447, p=0.00376, p_fdr=0.00794, N=42 ⚠ Cochran violations present (see §3.3.3)
   Yes vs No: χ²(3)=12.715, p=0.00530, p_fdr=0.00794, N=32 ⚠ Cochran violations present (see §3.3.3)

 Q35 - Why do you feel that way (open text)?
  [1] Das aktuelle Beschaffungskonzept des Staates vermittelt aktuell kein Sicherheitsgefühl 
  [2] Mehr Vertrauen in den Staat und weniger in Private
  [3] Korrekte Konfiguration bzgl. keine Daten übertragen  schwierig und aufwändig . Ich muss wissen wo meine und die Daten meiner Kunden liegen
  [4] Manipulationen durch staatliche Einflussnahme 
  [5] Ich vertraue dem Anbieter und den staatlichen Einrichtungen 
  [6] Auf Grund der proaktiven Information wäre man vor der Lage
  [7] Auslagerung an priv. Unternehmen keine Option
  [8] Niemand weiß, welche Personen dereinst den Staat repräsentieren werden und was dann ein Überwachunhssystem für Formen annehmen kann.
  [9] Aufgabe von Souveränität
  [10] Die Auslagerung einer SecIncident E

### Q36 [S7] - Trust in government agency (multi-dimensional)
>  **Survey question (S7):** "I trust a government agency regarding..."
>  *"Ich vertraue einer staatlichen Stelle hinsichtlich..."*
>
> **Dimensions:**
> [1] Quality of operations
> [2] Cybersecurity expertise
> [3] Data protection compliance  <- fear of violation = CONFIRMED FINDING #5
> [4] Compliance with agreed services
> [5] National cooperation  <- CONFIRMED FINDING #3
> [6] International cooperation
>
> **Scale:** Barely . Neutral . Strongly . Fully
>
>  **Q36[1] x Q22 = CONFIRMED FINDING #4** (cor_p=0.029, V=0.36)
>  **Q36[3] x Q22 = CONFIRMED FINDING #5** (cor_p=0.029, V=0.36)
>  **Q36[5] x Q22 = CONFIRMED FINDING #3** (cor_p=0.005, V=0.42)


In [31]:
fig_q36, df_q36 = get_bar('Q36', grp=False, db=False)
fig_q36.update_layout(title="Q36: Trust in government agency (all dimensions)")
fig_q36.show()

# FDR-BH batch over Q36[i] x Q22 (7 tests).
# Note: this applies FDR over this batch only — global confirmation across all 47 tests is in §8 / Cell 71.
q36_questions = [f'Q36[{i}]' for i in range(1, 8)]
print("\n Q36[i] x Q22 — FDR-BH over Q36 batch (global confirmation: §8 / Cell 71)")
print("Question: Does trust in government dimensions predict willingness to participate?")
res_q36 = batch_chi2_global_fdr(q36_questions, 'Q22', alpha=0.05, min_cell_n=10)
if len(res_q36):
    display(res_q36[['question','p_raw','p_fdr','reject_H0','N','cramers_v','cochran_ok']])

    # Show findings surviving this batch correction
    sig_q36 = res_q36[res_q36['reject_H0']]
    for _, row in sig_q36.iterrows():
        q = row['question']
        print(f"\n   {q}: '{q_dict.get(q,'')}'")
        fig_sig, _ = get_bar(q, 'Q22')
        fig_sig.update_layout(title=f" {q} x Q22 (confirmed finding, V={row['cramers_v']:.2f})")
        fig_sig.show()

# Open text: Q36EXT
print("\n Q36EXT - Other trust dimensions (open text):")
with pd.option_context('display.max_colwidth', 200):
    ot = usable_data[usable_data['Q36EXT'].notna()]['Q36EXT']
    for v in ot.values: print(f"  - {v}")



 Q36[i] x Q22 — FDR-BH over Q36 batch (global confirmation: §8 / Cell 71)
Question: Does trust in government dimensions predict willingness to participate?
χ²(6,N=64)=16.521, p=0.0112, V=0.359 ⚠ Cochran violations present (see §3.3.3) | Q36[3]xQ22


,question,p_raw,p_fdr,reject_H0,N,cramers_v,cochran_ok
0,Q36[3],0.01121,0.01121,True,64,0.3593,False



   Q36[3]: 'Ich vertraue einer staatlichen Stelle hinsichtlich [Einhaltung des Datenschutzes]'



 Q36EXT - Other trust dimensions (open text):
  - .
  - a
  - Kommerzielle Unternehmen
  - Schnelligkeit
  -  bin mir nicht ganz sicher, inwiefern Behörden hier getraut werden kann 
  - Es hat sich in der Vergangenheit gezeigt dass durch politische Postenbesetzungen die internationale Zusammenarbeit signifikant verschlechtert hat.
  - Yes
  - immer abhängig von der Stelle
  - Know How, Ressourcen
  - Compliance


### Q40 [S12] - Reasons for state involvement
>  **Survey question (S12):** "For what reason do you think the operation of such a solution might be of state interest?"
>  *"Aus welchem Grund denken Sie, könnte der Betrieb einer solchen Lösung von staatlichem Interesse sein?"*
>
> **Options (multi-select):**
> [1] Containment of economic damage
> [2] Better response in crisis situations
> [3] Records for criminal interest (embezzlement, corruption, tax evasion)  <- most sensitive
> [4] Containment of espionage activities
> [5] No opinion


In [32]:
fig_q40, df_q40 = get_bar('Q40', db=False)
fig_q40.update_layout(title="Q40: Perceived reasons for state SS-IDS interest (multi-select)")
fig_q40.show()

# Open text: Q40[other]
print("\n Q40[other] - Other perceived reasons:")
with pd.option_context('display.max_colwidth', 200):
    ot = usable_data[usable_data['Q40[other]'].notna()]['Q40[other]']
    for v in ot.values: print(f"  - {v}")



 Q40[other] - Other perceived reasons:
  - Kontrolle
  - Monitoring der gefahrenlage
  - Die Verbesserung der Sicherheit ist im staatlichen Interesse. Diese Systeme leisten einen wesentlichen Beitrag 
  - alle untentehenden und viele unredlich mehr!
  - Nur Informationen die von einem Sektor freigegeben werden dürfen dem Staat zur Verfügung gestellt werden
  - Überwachung
  - Frühwarnsystem für weitere Unternehmern und Branchen
  - Der Staat soll sich da raushalten


### Q28 [S29] - Security concerns about state-operated SS-IDS
>  **Survey question (S29):** "What would be potential concerns / security risks when the system is operated by the state?"
>  *"Was wären potenzielle Bedenken / Sicherheitsrisiken, wenn das System vom Staat betrieben wird?"*
>
> **Options (multi-select, key items):**
> [1] Data misuse by government
> [2] Monitoring/surveillance of employees
> [3] Unauthorized third-party access
> [4] Data leakage
> [5] System being targeted / hacked
> [6] Unencrypted data transmission
> [7] Lack of transparency about data processing
> [8] Legal uncertainty
>
> **Paper use (§5.3):** The distribution of concerns informs the policy recommendations.


In [33]:
fig_q28, df_q28 = get_bar('Q28', db=False)
fig_q28.update_layout(title="Q28: Security concerns about state-operated SS-IDS (multi-select)")
fig_q28.update_layout(xaxis={'categoryorder':'total descending'})
fig_q28.show()

# Cleaner chart (sorted by "High + Enormous")
plot_data_q28 = df_q28[df_q28['variable'] != 'Other (please specify)']
fig_q28b = px.bar(plot_data_q28, x='variable', y='Sizes', color='value',
    title="Q28: Security concerns (excl. Other, descending frequency)")
fig_q28b.update_layout(xaxis={'categoryorder':'total descending'})
fig_q28b.show()

# Open text: Q28EXT
print("\n Q28EXT - Other concerns (open text):")
with pd.option_context('display.max_colwidth', 200):
    ot = usable_data[usable_data['Q28EXT'].notna()]['Q28EXT']
    for v in ot.values: print(f"  - {v}")



 Q28EXT - Other concerns (open text):
  - Eigene Vertraulichkeitsverpflichtungen ggü Kunden müssen eingehalten werden.
  - Yes
  - Es muss eine neutrale Stelle sein, die staatlich gefördert werden sollte.
  - Der Staat der jetzt vertrauenswürdig ist muss es in 10 Jahen nicht unbedingt sein, aber könnte dann diese Lösung für seine Zwecke missbrauchen...


---
## Section 8 - Global FDR-BH Correction — Full 47-Test Battery

A single Benjamini–Hochberg FDR correction is applied simultaneously across all 47
pairwise tests. Only associations with FDR-adjusted p < 0.05 are classified as
statistically confirmed. This section shows the full battery and identifies the 5
surviving findings (KF1–KF5).

**Design note — n≥10 guard and FDR pool:** Raw p-values are collected for all 47
pre-specified pairs using `min_cell_n=1` (no pre-filtering) so that every pair
contributes to the FDR denominator. This is the conservative choice: including
small-group tests in the pool can only increase the FDR penalty, never reduce it.
The n≥10 minimum-group flag is then applied post-hoc to the sorted results:
pairs with any subgroup n < 10 are reported as descriptive only and excluded from
the confirmed-finding list. Individual analyses in §§4–7 use `min_cell_n=10` as
a pre-test guard, which is consistent — those cells test single pairs in isolation
rather than contributing to the global pool.

**BH validity under positive dependence:** The S7 trust sub-items (Q36[1]–Q36[7]) are
treated as independent pairs in the global FDR pool. BH controls FDR at level α under
independence and under positive regression dependence (PRDS condition); the trust sub-items
are expected to be positively correlated, satisfying this condition (Benjamini & Yekutieli 2001).

In [34]:
# Full 47-test battery (same pool as used for Table 2 in the paper)
full_battery = [
    ('Q22','Q34'), ('Q22','Q34Copy'), ('Q22','Q36[1]'), ('Q22','Q36[2]'),
    ('Q22','Q36[3]'), ('Q22','Q36[4]'), ('Q22','Q36[5]'), ('Q22','Q36[6]'), ('Q22','Q36[7]'),
    ('Q31','Q33'), ('Q31','Q08'), ('Q31','Q09'), ('Q31','Q25[1]'), ('Q31','Q25[2]'),
    ('Q31','Q25[3]'), ('Q31','Q25[4]'), ('Q31','Q25[5]'), ('Q31','Q25[6]'), ('Q31','Q36[1]'),
    ('Q32','Q33'), ('Q32','Q08'), ('Q32','Q09'), ('Q32','Q25[1]'), ('Q32','Q25[2]'),
    ('Q32','Q25[3]'), ('Q32','Q25[4]'), ('Q32','Q25[5]'), ('Q32','Q25[6]'),
    ('Q32','Q23[1]'), ('Q32','Q23[2]'), ('Q32','Q36[4]'),
    ('Q33','Q08'), ('Q33','Q09'), ('Q33','Q06'), ('Q33','Q07'), ('Q33','Q39'),
    ('Q34','Q25[1]'), ('Q34','Q25[2]'), ('Q34','Q25[3]'), ('Q34','Q25[4]'), ('Q34','Q25[5]'),
    ('Q34','Q25[6]'), ('Q34','Q36[1]'),
    ('Q37','Q08'), ('Q37','Q09'), ('Q37','Q34Copy'), ('Q37','Q36[1]'),
]

# Collect all raw p-values across the full 47-test battery (alpha=1.0, min_cell_n=1 to include all pairs).
# The n≥10 minimum-group guard is applied post-hoc (see Section 8 design note above).
raw_results = []
for q, sq in full_battery:
    r = test_phi(q, sq, alpha=1.0, db=False, min_cell_n=1)
    if r:
        # n≥10 minimum-group guard — flag pairs with any subgroup below threshold
        df_tmp = usable_data[[q, sq]].dropna()
        for d in df_drop['q']:  df_tmp = df_tmp[df_tmp[q] != d]
        for d in df_drop['sq']: df_tmp = df_tmp[df_tmp[sq] != d]
        row_n = df_tmp[q].value_counts()
        small = [v for v in df_tmp[q].unique() if row_n.get(v,0) < 10]
        r['n_blocked'] = len(small) > 0
        r['small_groups'] = small
        raw_results.append(r)

print(f"Tests in pool: {len(raw_results)}")

# Global FDR-BH
all_pvals = [r['p'] for r in raw_results]
reject_global, pvals_fdr, _, _ = multipletests(all_pvals, alpha=0.05, method='fdr_bh')
for r, rej, pfdr in zip(raw_results, reject_global, pvals_fdr):
    r['p_fdr'] = round(pfdr, 6)
    r['sig_global'] = bool(rej)

confirmed  = [r for r in raw_results if r['sig_global'] and not r['n_blocked']]
descriptive = [r for r in raw_results if not r['sig_global'] and not r['n_blocked'] and r['p'] < 0.05]
blocked    = [r for r in raw_results if r['n_blocked']]

print(f"\n{'='*60}")
print(f"CONFIRMED FINDINGS (global FDR-BH, n≥10 guard): {len(confirmed)}")
print(f"n<10 blocked — descriptive only:            {len(blocked)}")
print(f"Descriptive tendencies (raw p<0.05, not FDR): {len(descriptive)}")
print(f"{'='*60}")

out_confirmed = pd.DataFrame([{
    'Pair': f"{r['question']} x {r['subquestion']}",
    'χ²': round(r['chi2'],3), 'df': r['dof'],
    'p_raw': round(r['p'],5), 'cor_p (FDR)': r['p_fdr'],
    'N': r['N'], "Cramér's V": r['cramers_v'],
    'Effect': 'Large' if r['cramers_v']>=0.4 else ('Medium' if r['cramers_v']>=0.2 else 'Small'),
    'Cochran ': r['cochran_all_ge1'] and r['cochran_pct5_ok']
} for r in confirmed]).sort_values('cor_p (FDR)')
display(out_confirmed)

χ²(6,N=60)=21.690, p=0.0014, V=0.425 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34
χ²(6,N=62)=31.546, p=0.0000, V=0.504 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34Copy
χ²(6,N=64)=16.601, p=0.0109, V=0.360 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[1]
χ²(6,N=64)=11.508, p=0.0739, V=0.300 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[2]


χ²(6,N=64)=16.521, p=0.0112, V=0.359 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[3]


χ²(6,N=64)=12.570, p=0.0504, V=0.313 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[4]


χ²(6,N=63)=22.493, p=0.0010, V=0.423 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[5]
χ²(6,N=62)=10.053, p=0.1225, V=0.285 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[6]
χ²(4,N=8)=5.333, p=0.2548, V=0.577 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[7]


χ²(12,N=75)=30.841, p=0.0021, V=0.370 ⚠ Cochran violations present (see §3.3.3) | Q31xQ33


...

χ²(8,N=66)=18.078, p=0.0206, V=0.370 ⚠ Cochran violations present (see §3.3.3) | Q33xQ06
χ²(16,N=75)=22.239, p=0.1356, V=0.272 ⚠ Cochran violations present (see §3.3.3) | Q33xQ07
χ²(8,N=72)=31.961, p=0.0001, V=0.471 ⚠ Cochran violations present (see §3.3.3) | Q33xQ39
χ²(6,N=45)=13.091, p=0.0416, V=0.381 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[1]


χ²(6,N=47)=12.847, p=0.0455, V=0.370 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[2]


χ²(6,N=47)=11.876, p=0.0648, V=0.355 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[3]
χ²(6,N=47)=22.840, p=0.0009, V=0.493 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[4]
χ²(6,N=47)=25.049, p=0.0003, V=0.516 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[5]
χ²(2,N=3)=3.000, p=0.2231, V=1.000 ⚠ Cochran violations present (see §3.3.3) | Q34xQ25[6]
χ²(9,N=78)=26.281, p=0.0018, V=0.335 ⚠ Cochran violations present (see §3.3.3) | Q34xQ36[1]
χ²(9,N=72)=21.309, p=0.0113, V=0.314 ⚠ Cochran violations present (see §3.3.3) | Q37xQ08


χ²(9,N=73)=5.247, p=0.8123, V=0.155 ⚠ Cochran violations present (see §3.3.3) | Q37xQ09
χ²(9,N=70)=27.074, p=0.0014, V=0.359 ⚠ Cochran violations present (see §3.3.3) | Q37xQ34Copy
χ²(9,N=75)=21.821, p=0.0095, V=0.311 ⚠ Cochran violations present (see §3.3.3) | Q37xQ36[1]
Tests in pool: 47

CONFIRMED FINDINGS (global FDR-BH, n≥10 guard): 5
n<10 blocked — descriptive only:            37
Descriptive tendencies (raw p<0.05, not FDR): 1


,Pair,χ²,df,p_raw,cor_p (FDR),N,Cramér's V,Effect,Cochran
1,Q22 x Q34Copy,31.546,6,0.00002,0.000937,62,0.5044,Large,False
0,Q22 x Q34,21.690,6,0.00138,0.009252,60,0.4251,Large,False
4,Q22 x Q36[5],22.493,6,0.00099,0.009252,63,0.4225,Large,False
2,Q22 x Q36[1],16.601,6,0.01087,0.029630,64,0.3601,Medium,False
3,Q22 x Q36[3],16.521,6,0.01121,0.029630,64,0.3593,Medium,False


In [35]:
# Verification: confirm that none of the 5 confirmed findings appear in the blocked list.
# KF1–KF5 all involve Q22 (participation willingness) crossed with trust/security items.
kf_pairs = {('Q22','Q34Copy'), ('Q22','Q34'), ('Q22','Q36[3]'), ('Q22','Q36[4]'), ('Q22','Q36[5]')}
blocked_pairs = {(r['question'], r['subquestion']) for r in blocked}
overlap = kf_pairs & blocked_pairs

if not overlap:
    print("✓ Verification passed: none of KF1–KF5 appear in the n<10 blocked list.")
    print(f"  KF pairs:     {sorted(kf_pairs)}")
    print(f"  Blocked pairs (n<10): {len(blocked_pairs)} pairs, none overlapping with KF1–KF5.")
else:
    print(f"✗ WARNING: {len(overlap)} KF pair(s) also appear in the blocked list — review required.")
    print(f"  Overlap: {overlap}")

✓ Verification passed: none of KF1–KF5 appear in the n<10 blocked list.
  KF pairs:     [('Q22', 'Q34'), ('Q22', 'Q34Copy'), ('Q22', 'Q36[3]'), ('Q22', 'Q36[4]'), ('Q22', 'Q36[5]')]
  Blocked pairs (n<10): 37 pairs, none overlapping with KF1–KF5.


---
## Section 9 - The 5 Confirmed Findings

After global FDR-BH correction (47 tests), n≥10 guard, and Cochran check:

| # | Pair | cor_p | Cramer's V | Effect |
|---|------|-------|------------|--------|
| KF1 | Q22 [S21] x Q34Copy [S4] | 0.002 | 0.50 | Large |
| KF2 | Q22 [S21] x Q34 [S5] | 0.005 | 0.43 | Large |
| KF3 | Q22 [S21] x Q36[5] [S7] | 0.005 | 0.42 | Large |
| KF4 | Q22 [S21] x Q36[1] [S7] | 0.029 | 0.36 | Medium |
| KF5 | Q22 [S21] x Q36[3] [S7] | 0.029 | 0.36 | Medium |

Q32 x Q23[1]: raw p=0.035, cor_p=0.065 -> not confirmed, descriptive only.

Post-hoc category selection for each confirmed finding uses adjusted
standardized residuals (|z| > 2, interpretive only) —
no correction applied to residuals themselves (cf. §3.3.3).

**Cochran violations (documented limitation):** All 5 confirmed tests are 3×4 contingency
tables (12 cells). In each, 6–8 cells (50–67%) have expected frequency <5.
KF4 additionally has 2 cells below 1 (min = 0.61), driven by the sparse “No”
category in Q22 (n=14). All tests have N≥60; χ² is retained for pipeline
consistency (no single alternative method applies uniformly). These associations
should be interpreted as **indicative rather than confirmatory** (cf. §3.3.3 and §5).


In [36]:
# Visual summary: all 5 confirmed findings side by side
findings = [
    ('Q22','Q34Copy','KF1 - Feelings about state operation (V=0.50)'),
    ('Q22','Q34',    'KF2 - Feelings about commercial operation (V=0.43)'),
    ('Q22','Q36[5]', 'KF3 - Trust: national cooperation (V=0.42)'),
    ('Q22','Q36[1]', 'KF4 - Trust: operational quality (V=0.36)'),
    ('Q22','Q36[3]', 'KF5 - Trust: data protection compliance (V=0.36)'),
]
for q, sq, title in findings:
    fig, df_tmp = get_bar(q, sq)
    fig.update_layout(title=f" {title}  |  {q} x {sq}")
    fig.show()
    # Post-hoc driving categories via adjusted standardized residuals (|z|>2, interpretive only)
    pivot_tmp = df_tmp.pivot_table(
        index=q, columns=sq, values='count', aggfunc='sum').fillna(0)
    driving = get_driving_categories(pivot_tmp, z_threshold=2.0)
    if len(driving):
        print(f"   Driving categories (|z|>2) for {q} x {sq}:")
        display(driving[["row_category","col_category","observed","expected","std_residual_z"]])
    else:
        print(f"   No cells with |z|>2 for {q} x {sq}")

print()
print("Note: All 5 confirmed findings have Cochran violations (50-67% of cells with")
print("expected frequency <5). These associations are indicative rather than confirmatory.")
print("See §3.3.3 for full documentation.")


   Driving categories (|z|>2) for Q22 x Q34Copy:


,row_category,col_category,observed,expected,std_residual_z
0,No,Very insecure,7,1.92,4.394
1,No,Secure,0,5.18,-3.189
2,Don't know,Very insecure,1,4.52,-2.408
3,Other:,Very insecure,2,0.55,2.172
4,No,Insecure,7,3.84,2.109
5,Yes,Very insecure,0,2.60,-2.019


   Driving categories (|z|>2) for Q22 x Q34:


,row_category,col_category,observed,expected,std_residual_z
0,No,Very insecure,8,2.49,4.279
1,Don't know,Very insecure,2,5.88,-2.383


   Driving categories (|z|>2) for Q22 x Q36[5]:


...

   Driving categories (|z|>2) for Q22 x Q36[1]:


,row_category,col_category,observed,expected,std_residual_z
0,No response,Fully,2,0.21,4.189
1,No,Strongly,1,5.37,-2.672
2,No,Barely,5,1.92,2.665
3,Other:,Barely,2,0.55,2.172
4,Don't know,Strongly,17,12.66,2.100


   Driving categories (|z|>2) for Q22 x Q36[3]:


,row_category,col_category,observed,expected,std_residual_z
0,No,Barely,6,2.30,2.967
1,Other:,Neutral,3,0.82,2.772
2,Yes,Neutral,0,3.90,-2.577
3,No,Fully,0,3.26,-2.293



Note: All 5 confirmed findings have Cochran violations (50-67% of cells with
expected frequency <5). These associations are indicative rather than confirmatory.
See §3.3.3 for full documentation.


---
## Section 10 - Open-Text Responses


In [37]:
colwidth = 300
print("="*60)
print("Q24 - Why would your organization use SS-IDS services?")
print("="*60)
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q24'].notna()]['Q24']
    for i, v in enumerate(ot.values):
        print(f"  [{i+1:02d}] {v}")

print()
print("="*60)
print("Q26EXT - Other participation requirements")
print("="*60)
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q26EXT'].notna()]['Q26EXT']
    for v in ot.values: print(f"  - {v}")

print()
print("="*60)
print("Q25EXT - Other expected benefits")
print("="*60)
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q25EXT'].notna()]['Q25EXT']
    for v in ot.values: print(f"  - {v}")

print()
print("="*60)
print("Q17EXT - Other MSSP selection criteria")
print("="*60)
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q17EXT'].notna()]['Q17EXT']
    for v in ot.values: print(f"  - {v}")

print()
print("="*60)
print("Q28EXT - Other security concerns (state operation)")
print("="*60)
with pd.option_context('display.max_colwidth', colwidth):
    ot = usable_data[usable_data['Q28EXT'].notna()]['Q28EXT']
    for v in ot.values: print(f"  - {v}")


Q24 - Why would your organization use SS-IDS services?
  [01] Wäre hilfreich, wenn richtig gemacht...
  [02] Auf Grund der schlanken IT Besetzung (personell)
  [03] Staatliche Organisation
  [04] Ist eine Behörde
  [05] Profit von fremden Wissen
  [06] N/A aus Datenschutzgründen
  [07] Für den Informationsaustausch
  [08] Bundesheer muss sicher sein
  [09] meine Organisation würde es sehr umfangreich und sicher fühlen
  [10] ergänzende maßnahmen zu bereits bestehenden internen maßnahmen 
  [11] Sicherheit 
  [12] Da es im Bundesheer sehr viel über Erkennung und reaktion geht
  [13] Sammlung von Praktischer Erfahrung, Know-How Aufbau, Lehre
  [14] Bessere Erkennung von Angriffen und wenn sie funktioniert weniger Mitarbeiter notwendig 
  [15] Weil wir es brauchen
  [16] Ich denke man muss "Erkennung" und "Reaktion" vermischen.
Für die "Reaktion" ist die Unerstützung einer staatlichen Stelle extrem hilfreich. 
Das Herumschnüffeln im Innenbereich der IT unter dem Deckmantel der "Erkennung"

---
## Section 11 - Soundness Check


In [38]:
print("="*60)
print("SOUNDNESS CHECKS")
print("="*60)

print(f"\n1. Sample size (should be 89): N = {total_part}")

# Check Q22 distribution
fig_q22_check, df_q22_check = get_bar('Q22', db=False)
# Q22 actual options: Yes / No / Don't know (+ No response / Other: dropped)
yes_pct  = df_q22_check[df_q22_check['unique_values']=='Yes']['counts'].sum() / total_part * 100
no_pct   = df_q22_check[df_q22_check['unique_values']=='No']['counts'].sum() / total_part * 100
dk_pct   = df_q22_check[df_q22_check['unique_values']=="Don't know"]['counts'].sum() / total_part * 100
print(f"\n2. Q22 distribution (actual options: Yes/No/Don't know):")
print(f"   Yes:            {yes_pct:.1f}%")
print(f"   No:             {no_pct:.1f}%")
print(f"   Don't know:     {dk_pct:.1f}%")
print(f"   Sum (Y+N+DK):   {yes_pct+no_pct+dk_pct:.1f}% (excl. No-response/Other)")

print(f"\n3. Confirmed findings - verify cor_p values:")
expected_findings = [
    ('Q22','Q34Copy', 0.002, 0.50),
    ('Q22','Q34',     0.005, 0.43),
    ('Q22','Q36[5]',  0.005, 0.42),
    ('Q22','Q36[1]',  0.029, 0.36),
    ('Q22','Q36[3]',  0.029, 0.36),
]
for q, sq, exp_p, exp_v in expected_findings:
    r = test_phi(q, sq, alpha=1.0, db=False, min_cell_n=1)
    if r:
        status = "" if abs(r['cramers_v'] - exp_v) < 0.05 else "️"
        print(f"   {status} {q}x{sq}: V={r['cramers_v']:.3f} (expected ~{exp_v}), raw p={r['p']:.4f}")

print(f"\n4. Q32xQ23[1] should be demoted after global FDR (cor_p~0.065, not confirmed):")
r32 = test_phi('Q32', 'Q23[1]', alpha=1.0, db=False, min_cell_n=1)
if r32:
    print(f"   raw p = {r32['p']:.5f} (needs global FDR to get cor_p — full battery run in §9 (Confirmed Findings))")

print(f"\n5. HTL label fix in Q10 (should read Upper Secondary Technical School):")
fig_q10c, df_q10c = get_bar('Q10', db=False)
htl_rows = df_q10c[df_q10c['unique_values'].str.contains('HTL|Secondary|Associate', na=False)]
print(f"   HTL label: {list(htl_rows['unique_values'])}")
print(f"   {' Correctly shows Secondary/HTL' if 'Associate' not in str(list(htl_rows['unique_values'])) else '️ Still shows Associate Degree'}")


SOUNDNESS CHECKS

1. Sample size (should be 89): N = 89

2. Q22 distribution (actual options: Yes/No/Don't know):
   Yes:            21.3%
   No:             15.7%
   Don't know:     37.1%
   Sum (Y+N+DK):   74.2% (excl. No-response/Other)

3. Confirmed findings - verify cor_p values:
χ²(6,N=62)=31.546, p=0.0000, V=0.504 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34Copy
    Q22xQ34Copy: V=0.504 (expected ~0.5), raw p=0.0000


χ²(6,N=60)=21.690, p=0.0014, V=0.425 ⚠ Cochran violations present (see §3.3.3) | Q22xQ34
    Q22xQ34: V=0.425 (expected ~0.43), raw p=0.0014
χ²(6,N=63)=22.493, p=0.0010, V=0.423 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[5]
    Q22xQ36[5]: V=0.422 (expected ~0.42), raw p=0.0010


χ²(6,N=64)=16.601, p=0.0109, V=0.360 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[1]
    Q22xQ36[1]: V=0.360 (expected ~0.36), raw p=0.0109


χ²(6,N=64)=16.521, p=0.0112, V=0.359 ⚠ Cochran violations present (see §3.3.3) | Q22xQ36[3]
    Q22xQ36[3]: V=0.359 (expected ~0.36), raw p=0.0112

4. Q32xQ23[1] should be demoted after global FDR (cor_p~0.065, not confirmed):


χ²(1,N=46)=4.696, p=0.0302, V=0.320 | Q32xQ23[1]
   raw p = 0.03024 (needs global FDR to get cor_p — full battery run in §9 (Confirmed Findings))

5. HTL label fix in Q10 (should read Upper Secondary Technical School):


   HTL label: []
    Correctly shows Secondary/HTL


---
## Section 12 — Paper Figures (PDF export)

Regenerate all four figures used in the paper.
Figures are saved to `../stateids___TOPS_2026/Auswertung/` as v2 PDFs.

| Figure | File | Section |
|--------|------|---------|
| Fig 2 | `imprtance_of_ti_dodged_bar_chart-v2.pdf` | §5.1 — Importance of D&R systems |
| Fig 3 | `participant_motivation_dodged_bar_chart-v2.pdf` | §5.1 — Expected benefits (Q25) |
| Fig 4 | `govtrust_dodged_bar_chart-v2.pdf` | §5.3 — Government trust (Q36) |
| Fig 5 | `security_risks_dodged_bar_chart-v2.pdf` | §5.3 — Security concerns (Q28) |


In [39]:
# ── Fig 2: Importance of D&R systems (Q31/Q32/Q33) ────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pltq31 = get_bar('Q31',db=False)[1][['unique_values','counts']].rename({'unique_values':'value', 'counts':'Sizes'}, axis=1)
pltq31['variable'] = 'Importance of Detect & Response'
pltq32 = get_bar('Q32',db=False)[1][['unique_values','counts']].rename({'unique_values':'value', 'counts':'Sizes'}, axis=1)
pltq32['variable'] = 'Perceived Increase of Security'
pltq33 = get_bar('Q33',db=False)[1][['unique_values','counts']].rename({'unique_values':'value', 'counts':'Sizes'}, axis=1).replace(
    {'We do not use such systems':'Not in use', 'No response':'Not disclosed'})
pltq33['variable'] = 'Benefits for the Org.'

plot_data = pd.concat([pltq31, pltq32, pltq33])
pivot_data = plot_data.pivot(index='variable', columns='value', values='Sizes').fillna(0)
pivot_data.index = pivot_data.index.to_series().apply(lambda x: f'{x} [{int(pivot_data.loc[x].sum())}]')
pivot_data_percentage = pivot_data.div(pivot_data.sum(axis=1), axis=0) * 100
pivot_data_sorted = pivot_data_percentage.sort_values(by='Very important', ascending=True)

patterns = ['\\\\', 'x', '*', '///', 'xx']
colors = ['#1e8449','#009e73', '#e69f00', '#d35400', '#c0392b', '#cacfd2']
fig2, ax2 = plt.subplots(figsize=(3, 1.5))
col_order2 = [c for c in ['Very important','Important','Neutral','Not important','Not in use','Not disclosed']
              if c in pivot_data_sorted.columns]
pivot_data_sorted[col_order2].plot(kind='barh', stacked=True, color=colors[:len(col_order2)], ax=ax2)
plt.xlabel('Number of responses')
plt.ylabel('Perc. Importance', labelpad=20)
pl2 = []
for p in patterns[:len(col_order2)]: pl2 += [p] * len(pivot_data_sorted)
for bar, pattern in zip(ax2.patches, pl2): bar.set_hatch(pattern)
plt.legend(bbox_to_anchor=(-0.1, -0.4), loc='upper center', ncol=4, fancybox=False, shadow=False)
_out2 = '../stateids___TOPS_2026/Auswertung/imprtance_of_ti_dodged_bar_chart-v2.pdf'
plt.savefig(_out2, format='pdf', bbox_inches='tight'); plt.close()
print(f'✓ Fig 2 saved: {_out2}')

✓ Fig 2 saved: ../stateids___TOPS_2026/Auswertung/imprtance_of_ti_dodged_bar_chart-v2.pdf


In [40]:
# ── Fig 3: Expected benefits / Q25 ─────────────────────────────────────────────
plot_data = get_bar('Q25',db=False)[1]
plot_data = plot_data[plot_data['variable']!='Other (please specify):']
pivot_data = plot_data.pivot(index='variable', columns='value', values='Sizes').fillna(0)
pivot_data = pivot_data.rename(index={
    'A better internal exchange':'Better internal exch.',
    'A better overview of the current threat situation in the sector and across sectors':'Better overview threat situation',
    'Access to cost-effective threat intelligence feeds':'Access to threat intel.',
    'Additional external expertise':'Additional external expertise',
    'An additional communication platform for rapid exchange with other affected organizations':'Add. communciation plat.'
})
pivot_data.index = pivot_data.index.to_series().apply(lambda x: f'{x} [{pivot_data.loc[x].sum()}]')
pivot_data_percentage = pivot_data.div(pivot_data.sum(axis=1), axis=0) * 100
pivot_data_sorted = pivot_data_percentage.sort_values(by='Very important', ascending=True)
patterns = ['', '///', '\\\\', 'xx']
colors = ['#e69f00', '#56b4e9', '#009e73', '#f0e442']
fig3, ax3 = plt.subplots(figsize=(3, 1.5))
pivot_data_sorted[['Very important', 'Desirable', 'Irrelevant']].plot(kind='barh', stacked=True, color=colors, ax=ax3)
plt.xlabel('Number of responses')
plt.ylabel('Req. functionality', labelpad=20)
pl3 = []
for p in patterns: pl3 += [p] * len(pivot_data_sorted)
for bar, pattern in zip(ax3.patches, pl3): bar.set_hatch(pattern)
plt.legend(title='Importance', bbox_to_anchor=(-0.1, -0.4), loc='upper center', ncol=4, fancybox=False, shadow=False)
_out3 = '../stateids___TOPS_2026/Auswertung/participant_motivation_dodged_bar_chart-v2.pdf'
plt.savefig(_out3, format='pdf', bbox_inches='tight'); plt.close()
print(f'✓ Fig 3 saved: {_out3}')

✓ Fig 3 saved: ../stateids___TOPS_2026/Auswertung/participant_motivation_dodged_bar_chart-v2.pdf


In [41]:
# ── Fig 4: Government trust dimensions / Q36 ───────────────────────────────────
plot_data = get_bar('Q36',grp=False,db=False)[1]
plot_data = plot_data[plot_data['variable']!='Other (please specify):']
pivot_data = plot_data.pivot(index='variable', columns='value', values='Sizes').fillna(0)
pivot_data = pivot_data.rename(index={'Adherence to service level agreements':'Adherence to SLAs'})
pivot_data.index = pivot_data.index.to_series().apply(lambda x: f'{x} [{pivot_data.loc[x].sum()}]')
pivot_data_percentage = pivot_data.div(pivot_data.sum(axis=1), axis=0) * 100
pivot_data_sorted = pivot_data_percentage.sort_values(by='Barely', ascending=True)
patterns = ['', '///', '\\\\', 'xx', '+']
colors = ['#CC0033', '#56b4e9', '#90EE90', '#009e73', '#f0e442']
fig4, ax4 = plt.subplots(figsize=(4, 2))
pivot_data_sorted[['Barely','Neutral','Strongly','Fully','No response']].plot(kind='barh', stacked=True, color=colors, ax=ax4)
ax4.set_xlabel('Percentage of responses')
ax4.set_ylabel('Gov. capabilities')
pl4 = []
for p in patterns: pl4 += [p] * len(pivot_data_sorted)
for bar, pattern in zip(ax4.patches, pl4): bar.set_hatch(pattern)
plt.legend(title='Trust level', bbox_to_anchor=(-0, -0.35), loc='upper center', ncol=5, fancybox=True, shadow=False)
_out4 = '../stateids___TOPS_2026/Auswertung/govtrust_dodged_bar_chart-v2.pdf'
plt.savefig(_out4, format='pdf', bbox_inches='tight'); plt.close()
print(f'✓ Fig 4 saved: {_out4}')

✓ Fig 4 saved: ../stateids___TOPS_2026/Auswertung/govtrust_dodged_bar_chart-v2.pdf


In [42]:
# ── Fig 5: Security concerns / Q28 ─────────────────────────────────────────────
plot_data = get_bar('Q28',db=False)[1]
plot_data = plot_data[plot_data['variable']!='Other (please specify):']
pivot_data = plot_data.pivot(index='variable', columns='value', values='Sizes').fillna(0)
pivot_data = pivot_data.rename(index={
    'Abuse, corruption concerns, economic espionage through data access':'Abuse, espionage via data access',
    'Breaking encrypted connections':'Breaking encrypted conn.',
    'Centralization of data':'Data centralization',
    'Competition with commercial service providers':'Competition w. commercial',
    'Concealment of information (attacks) for state interests':'Concealment of attacks',
    'Exposure of private or sensitive information':'Exposure of sensitive info.',
    'Lack of transparency and documentation':'Lack of transparency',
    'Monitoring of employees':'Monitoring of employees',
    'Non-compliance with contractual agreements':'Non-compliance w. aggreements',
    'Official principle / close relationship with law enforcement (reporting of criminally relevant events)':'Relationship w. law enforcement',
    'Sale of data':'Sale of data',
    'Vulnerabilities and security risks of the solution itself':'Vuln. and risks of solution'
})
pivot_data.index = pivot_data.index.to_series().apply(lambda x: f'{x} [{pivot_data.loc[x].sum()}]')
pivot_data_percentage = pivot_data.div(pivot_data.sum(axis=1), axis=0) * 100
pivot_data_sorted = pivot_data_percentage.sort_values(by='Enormous', ascending=True)
patterns = ['', '///', '\\\\', 'xx']
colors = ['#e69f00', '#56b4e9', '#009e73', '#f0e442']
fig5, ax5 = plt.subplots(figsize=(3, 3))
pivot_data_sorted[['Enormous','High','Low','No response']].plot(kind='barh', stacked=True, color=colors, ax=ax5)
ax5.set_xlabel('Percentage of responses')
ax5.set_ylabel('Security risks')
pl5 = []
for p in patterns: pl5 += [p] * len(pivot_data_sorted)
for bar, pattern in zip(ax5.patches, pl5): bar.set_hatch(pattern)
plt.legend(title='Risk level', bbox_to_anchor=(-0.1, -0.25), loc='upper center', ncol=4, fancybox=True, shadow=False)
_out5 = '../stateids___TOPS_2026/Auswertung/security_risks_dodged_bar_chart-v2.pdf'
plt.savefig(_out5, format='pdf', bbox_inches='tight'); plt.close()
print(f'✓ Fig 5 saved: {_out5}')

✓ Fig 5 saved: ../stateids___TOPS_2026/Auswertung/security_risks_dodged_bar_chart-v2.pdf


---
## Notebook complete

Run all cells top to bottom in a single kernel session.

**Dependencies:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `plotly`, `scipy`, `statsmodels`

**Data files required:**
- `survey/results-survey736756.csv` — anonymised survey responses
- `survey/sentences_translated.csv` — German-to-English translation map

See `README.md` for full setup and execution instructions.